In [1]:
!git clone https://github.com/CryAndRRich/visenet.git

Cloning into 'visenet'...
remote: Enumerating objects: 334, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 334 (delta 12), reused 11 (delta 7), pack-reused 312 (from 1)
Receiving objects: 100% (334/334), 176.83 MiB | 34.16 MiB/s, done.
Resolving deltas: 100% (159/159), done.


In [11]:
!pip install -r visenet/requirements.txt

  Using cached matplotlib-3.9.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (11 kB)
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached scipy-1.14.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
  Using cached torch-2.3.1-cp312-cp312-manylinux1_x86_64.whl.metadata (26 kB)
  Using cached torchvision-0.20.1-cp312-cp312-manylinux1_x86_64.whl.metadata (6.1 kB)
  Using cached stable_baselines3-2.7.0-py3-none-any.whl.metadata (4.8 kB)
  Using cached sb3_contrib-2.7.0-py3-none-any.whl.metadata (4.1 kB)
  Using cached Shimmy-2.0.0-py3-none-any.whl.metadata (3.5 kB)
  Using cached pandas_market_calendars-5.1.1-py3-none-any.whl.metadata (9.7 kB)
  Using cached pathlib-1.0.1-py3-none-any.whl.metadata (5.1 kB)
  Using cached streamlit-1.49.1-py3-none-any.whl.metadata (9.5 kB)
  Using cached plotly-6.0.0-py3-none-any.whl.metadata (5.6 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.

In [12]:
!pip install stable-baselines3

  Using cached stable_baselines3-2.7.0-py3-none-any.whl.metadata (4.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.2/187.2 kB 4.1 MB/s eta 0:00:00


In [3]:
import torch
print(torch.__version__)

2.8.0+cu126


In [4]:
import sys
sys.path.append("/content/visenet")


In [5]:
def data_split(df, start, end):
    """Tách dữ liệu thành tập huấn luyện hoặc kiểm tra dựa trên ngày tháng"""
    data = df[(df.timestamp >= start) & (df.timestamp < end)]
    data=data.sort_values(['timestamp', 'ticker'], ignore_index=True)
    data.index = data.timestamp.factorize()[0]
    return data

In [6]:
!unzip /content/visenet/backtesting/trained_models/trained_models.zip -d /content/results

Archive:  /content/visenet/backtesting/trained_models/trained_models.zip
   creating: /content/results/trained_models/
   creating: /content/results/trained_models/2025-09-02 01:44:37.827718/
 extracting: /content/results/trained_models/2025-09-02 01:44:37.827718/TD3_10k_dow_252.zip  
 extracting: /content/results/trained_models/2025-09-02 01:44:37.827718/TD3_10k_dow_378.zip  
 extracting: /content/results/trained_models/2025-09-02 01:44:37.827718/A2C_30k_dow_252.zip  
 extracting: /content/results/trained_models/2025-09-02 01:44:37.827718/PPO_100k_dow_189.zip  
 extracting: /content/results/trained_models/2025-09-02 01:44:37.827718/A2C_30k_dow_189.zip  
 extracting: /content/results/trained_models/2025-09-02 01:44:37.827718/A2C_30k_dow_126.zip  
 extracting: /content/results/trained_models/2025-09-02 01:44:37.827718/PPO_100k_dow_252.zip  
 extracting: /content/results/trained_models/2025-09-02 01:44:37.827718/TD3_10k_dow_315.zip  
 extracting: /content/results/trained_models/2025-09-0

In [13]:
# ================================================================
# Common libraries
# ================================================================
import pandas as pd
import numpy as np
import time
import gym

# ================================================================
# RL models from stable-baselines3
# ================================================================
from stable_baselines3 import A2C, PPO, TD3, SAC
from stable_baselines3.common.noise import OrnsteinUhlenbeckActionNoise
from stable_baselines3.common.vec_env import DummyVecEnv

# ================================================================
# Customized env
# ================================================================
from env.EnvMultipleStock_train import StockEnvTrain
from env.EnvMultipleStock_validation import StockEnvValidation
from env.EnvMultipleStock_trade import StockEnvTrade
from config import config
#from preprocess.preprocessor import data_split
# ================================================================
# Training functions
# ================================================================
STOCK_DIM = 30
def train_A2C(env_train, model_name, timesteps=25000):
    """Train A2C model"""
    start = time.time()
    model = A2C("MlpPolicy", env_train, verbose=0)
    model.learn(total_timesteps=timesteps)
    end = time.time()
    model.save(f"{config.TRAINED_MODEL_DIR}/{model_name}")
    print("Training time (A2C): ", (end - start) / 60, " minutes")
    return model


def train_TD3(env_train, model_name, timesteps=10000):
    """Train TD3 model (thay cho DDPG)"""
    n_actions = env_train.action_space.shape[-1]
    action_noise = OrnsteinUhlenbeckActionNoise(
        mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions)
    )

    start = time.time()
    model = TD3("MlpPolicy", env_train, action_noise=action_noise, verbose=0)
    model.learn(total_timesteps=timesteps)
    end = time.time()
    model.save(f"{config.TRAINED_MODEL_DIR}/{model_name}")
    print("Training time (TD3): ", (end - start) / 60, " minutes")
    return model


def train_PPO(env_train, model_name, timesteps=50000):
    """Train PPO model"""
    start = time.time()
    model = PPO("MlpPolicy", env_train, ent_coef=0.005, verbose=0)
    model.learn(total_timesteps=timesteps)
    end = time.time()
    model.save(f"{config.TRAINED_MODEL_DIR}/{model_name}")
    print("Training time (PPO): ", (end - start) / 60, " minutes")
    return model

# ================================================================
# DRL prediction / validation
# ================================================================
from datetime import datetime
import numpy as np

def DRL_prediction(model, environment, test_data, test_env, test_obs):
    """
    model: DRL model (PPO/A2C/TD3)
    environment: dataframe gốc (environment data) - có thể dùng nếu cần
    test_data: dataframe trading hôm nay
    test_env: môi trường StockEnvTrade
    test_obs: observation từ env_trade.reset()
    """
    account_memory = []
    actions_memory = []
    daily_actions = []

    obs = test_obs

    # Lấy danh sách ngày thực tế từ cột timestamp
    if 'timestamp' in test_data.columns:
        unique_trade_date = test_data['timestamp'].unique()
    else:
        # fallback nếu không có cột timestamp
        unique_trade_date = test_data.index.unique()

    for date in unique_trade_date:
        # Predict action
        action, _states = model.predict(obs)
        obs, rewards, dones, info = test_env.step(action)

        actions_memory.append(action)
        account_memory.append(test_env.env_method("save_asset_memory")[0])

        # Format ngày sang dd-mm-yy
        try:
            formatted_date = datetime.strptime(str(date), "%Y%m%d").strftime("%d-%m-%y")
        except ValueError:
            # fallback nếu date đã là datetime
            formatted_date = pd.to_datetime(date).strftime("%d-%m-%y")

        daily_actions.append({
            "date": formatted_date,
            "action": np.array(action).flatten().tolist(),
            "stocks": test_env.envs[0].df.ticker.unique().tolist(),
            "cash": test_env.envs[0].state[0],
            "portfolio_value": test_env.envs[0].state[0] +
                               sum(np.array(test_env.envs[0].state[1:(STOCK_DIM + 1)]) *
                                   np.array(test_env.envs[0].state[(STOCK_DIM + 1):(STOCK_DIM * 2 + 1)]))
        })

        if dones:
            break

    return account_memory, actions_memory, daily_actions





def DRL_validation(model, test_data, test_env, test_obs) -> None:
    """Validation loop"""
    for i in range(len(test_data.index.unique())):
        action, _states = model.predict(test_obs)
        test_obs, rewards, dones, info = test_env.step(action)


def get_validation_sharpe(iteration):
    """Calculate Sharpe ratio from validation results"""
    df_total_value = pd.read_csv(
        f"results/account_value_validation_{iteration}.csv", index_col=0
    )
    df_total_value.columns = ["account_value_train"]
    df_total_value["daily_return"] = df_total_value.pct_change(1)
    sharpe = (4 ** 0.5) * df_total_value["daily_return"].mean() / \
             df_total_value["daily_return"].std()
    return sharpe

# ================================================================
# Ensemble strategy
# ================================================================
def run_ensemble_strategy(df, unique_trade_date, rebalance_window, validation_window,
                          to_email, from_email, app_password) -> None:
    """
    Ensemble Strategy combining PPO, A2C and TD3
    - Train A2C, PPO, TD3 trong mỗi rebalance window
    - Chọn model tốt nhất theo Sharpe
    - Thực hiện trading với model đó
    - Log chiến lược từng ngày (daily_actions)
    - Gửi email cảnh báo cuối mỗi ngày trading
    """

    print("============Start Ensemble Strategy============")

    ppo_sharpe_list = []
    td3_sharpe_list = []
    a2c_sharpe_list = []
    model_use = []

    insample_turbulence = df[(df.timestamp < 20240101) & (df.timestamp >= 20181009)]
    insample_turbulence = insample_turbulence.drop_duplicates(subset=["timestamp"])
    insample_turbulence_threshold = np.quantile(insample_turbulence.turbulence.values, .90)

    start = time.time()
    for i in range(rebalance_window + validation_window, len(unique_trade_date), rebalance_window):
        print("============================================")

        # === turbulence threshold ===
        end_date_index = df.index[df["timestamp"] ==
                                  unique_trade_date[i - rebalance_window - validation_window]].to_list()[-1]
        end_date_index = int(end_date_index)
        start_date_index = end_date_index - validation_window * 30 + 1
        historical_turbulence = df.iloc[start_date_index:(end_date_index + 1), :]
        historical_turbulence = historical_turbulence.drop_duplicates(subset=["timestamp"])
        historical_turbulence_mean = np.mean(historical_turbulence.turbulence.values)

        if historical_turbulence_mean > insample_turbulence_threshold:
            turbulence_threshold = insample_turbulence_threshold
        else:
            turbulence_threshold = np.quantile(insample_turbulence.turbulence.values, 1)
        print("turbulence_threshold: ", turbulence_threshold)

        # === training env ===
        train = data_split(df, start=20181009,
                           end=unique_trade_date[i - rebalance_window - validation_window])
        env_train = DummyVecEnv([lambda: StockEnvTrain(train)])

        # === validation env ===
        validation = data_split(df,
                                start=unique_trade_date[i - rebalance_window - validation_window],
                                end=unique_trade_date[i - rebalance_window])
        env_val = DummyVecEnv([lambda: StockEnvValidation(validation,
                                                          turbulence_threshold=turbulence_threshold,
                                                          iteration=i)])
        obs_val = env_val.reset()

        # Train A2C
        print("======A2C Training========")
        model_a2c = train_A2C(env_train, f"A2C_30k_dow_{i}", timesteps=30000)
        DRL_validation(model_a2c, validation, env_val, obs_val)
        sharpe_a2c = get_validation_sharpe(i)
        print("A2C Sharpe Ratio: ", sharpe_a2c)

        # Train PPO
        print("======PPO Training========")
        model_ppo = train_PPO(env_train, f"PPO_100k_dow_{i}", timesteps=100000)
        DRL_validation(model_ppo, validation, env_val, obs_val)
        sharpe_ppo = get_validation_sharpe(i)
        print("PPO Sharpe Ratio: ", sharpe_ppo)

        # Train TD3
        print("======TD3 Training========")
        model_td3 = train_TD3(env_train, f"TD3_10k_dow_{i}", timesteps=10000)
        DRL_validation(model_td3, validation, env_val, obs_val)
        sharpe_td3 = get_validation_sharpe(i)
        print("TD3 Sharpe Ratio: ", sharpe_td3)

        ppo_sharpe_list.append(sharpe_ppo)
        a2c_sharpe_list.append(sharpe_a2c)
        td3_sharpe_list.append(sharpe_td3)

        # === model selection ===
        if (sharpe_ppo >= sharpe_a2c) and (sharpe_ppo >= sharpe_td3):
            model_ensemble = model_ppo
            model_use.append("PPO")
        elif (sharpe_a2c > sharpe_ppo) and (sharpe_a2c > sharpe_td3):
            model_ensemble = model_a2c
            model_use.append("A2C")
        else:
            model_ensemble = model_td3
            model_use.append("TD3")

        # === Trading ===
        print("======Trading from: ", unique_trade_date[i - rebalance_window], "to ", unique_trade_date[i])
        trade = data_split(df,
                           start=unique_trade_date[i - rebalance_window],
                           end=unique_trade_date[i])
        env_trade = DummyVecEnv([lambda: StockEnvTrade(trade)])
        obs_trade = env_trade.reset()

        account_memory, actions_memory, daily_actions = DRL_prediction(
            model_ensemble, df, trade, env_trade, obs_trade
        )

        alerts = parse_actions(daily_actions)
        print("Alerts: ", alerts)
        # gửi email mỗi ngày
        for alert in alerts:
            send_email(
                subject="Cảnh báo giao dịch từ VISENET",
                body=alert,
                to_email=to_email,
                from_email=from_email,
                app_password=app_password
            )

    end = time.time()
    print("Ensemble Strategy took: ", (end - start) / 60, " minutes")




/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [68]:
import numpy as np
import pandas as pd

import gym
from gym import spaces
from gym.utils import seeding

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Mỗi lần giao dịch tối đa mua/bán 100 cổ phiếu
HMAX_NORMALIZE = 100
# Lượng tiền ban đầu
INITIAL_ACCOUNT_BALANCE = 1000000
# Số luợng cổ phiếu trong danh mục đầu tư
STOCK_DIM = 30
# Phí giao dịch
TRANSACTION_FEE_PERCENT = 0.001

# Chỉ số biến động: ngưỡng hợp lý 90-150
# TURBULENCE_THRESHOLD = 140
REWARD_SCALING = 1e-4

class StockEnvTrade(gym.Env):
    """Môi trường giao dịch chứng khoán cho OpenAI gym"""
    metadata = {"render.modes": ["human"]}

    def __init__(self,
                 df: pd.DataFrame,
                 day: int = 0,
                 turbulence_threshold: int = 140,
                 initial: bool = True,
                 previous_state = [],
                 model_name = "",
                 iteration = "") -> None:
        # super(StockEnv, self).__init__()

        # money = 10
        # scope = 1
        self.day = day
        self.df = df
        self.initial = initial
        self.previous_state = previous_state

        self.action_space = spaces.Box(low = -1, high = 1,shape = (STOCK_DIM,))

        # Số chiều = 181 = [Luoợng tiền hiện có] + [Giá đóng cửa điều chỉnh của 30 cổ phiếu] +
        # [Số cổ phiếu đang sở hữu của 30 cổ phiếu] + [MACD của 30 cổ phiếu] + [RSI của 30 cổ phiếu] +
        # [CCI của 30 cổ phiếu] + [ADX của 30 cổ phiếu]
        self.observation_space = spaces.Box(low=0, high=np.inf, shape=(181,))

        # Load dữ liệu
        self.data = self.df.loc[self.day,:]
        self.terminal = False
        self.turbulence_threshold = turbulence_threshold

        self.state = [INITIAL_ACCOUNT_BALANCE] + \
                      self.data.close.values.tolist() + \
                      [0] * STOCK_DIM + \
                      self.data.macd.values.tolist() + \
                      self.data.rsi.values.tolist() + \
                      self.data.cci.values.tolist() + \
                      self.data.adx.values.tolist()

        # Reward
        self.reward = 0
        self.turbulence = 0
        self.cost = 0
        self.trades = 0

        # Lưu trữ giá trị tài sản theo thời gian
        self.asset_memory = [INITIAL_ACCOUNT_BALANCE]
        self.rewards_memory = []
        # self.reset()
        self._seed()
        self.model_name = model_name
        self.iteration = iteration

    def _sell_stock(self, index, action):
        # Thực hiện hành động bán dựa trên dấu của hành động
        if self.turbulence < self.turbulence_threshold:
            if self.state[index + STOCK_DIM + 1] > 0:
                # Cập nhật số dư
                self.state[0] += self.state[index + 1] * min(abs(action), self.state[index + STOCK_DIM + 1]) * (1 - TRANSACTION_FEE_PERCENT)
                self.state[index + STOCK_DIM + 1] -= min(abs(action), self.state[index + STOCK_DIM + 1])
                self.cost += self.state[index + 1] * min(abs(action), self.state[index + STOCK_DIM + 1]) * TRANSACTION_FEE_PERCENT
                self.trades += 1
            else:
                pass
        else:
            # Nếu biến động vượt quá ngưỡng, xóa tất cả các vị trí
            if self.state[index + STOCK_DIM + 1] > 0:
                # Câp nhật số dư
                self.state[0] += self.state[index + 1] * self.state[index + STOCK_DIM + 1] * (1 - TRANSACTION_FEE_PERCENT)
                self.state[index + STOCK_DIM + 1] = 0
                self.cost += self.state[index + 1] * self.state[index + STOCK_DIM + 1] * TRANSACTION_FEE_PERCENT
                self.trades += 1
            else:
                pass

    def _buy_stock(self, index, action):
        # Thực hiện hành động mua dựa trên dấu của hành động
        if self.turbulence< self.turbulence_threshold:
            available_amount = self.state[0] // self.state[index + 1]
            # print("available_amount: {}".format(available_amount))

            # Cập nhật số dư
            self.state[0] -= self.state[index + 1] * min(available_amount, action) * (1 + TRANSACTION_FEE_PERCENT)

            self.state[index + STOCK_DIM + 1] += min(available_amount, action)

            self.cost += self.state[index + 1] * min(available_amount, action) * TRANSACTION_FEE_PERCENT
            self.trades += 1
        else:
            # Nếu biến động vượt quá ngưỡng, không mua cổ phiếu
            pass

    def step(self, actions):
        # print(self.day)
        self.terminal = self.day >= len(self.df.index.unique()) - 1
        # print(actions)

        if self.terminal:
            plt.plot(self.asset_memory, "r")
            plt.savefig("results/account_value_trade_{}_{}.png".format(self.model_name, self.iteration))
            plt.close()
            df_total_value = pd.DataFrame(self.asset_memory)
            df_total_value.to_csv("results/account_value_trade_{}_{}.csv".format(self.model_name, self.iteration))
            end_total_asset = self.state[0] + sum(np.array(self.state[1:(STOCK_DIM + 1)]) * np.array(self.state[(STOCK_DIM + 1):(STOCK_DIM * 2 + 1)]))
            print("previous_total_asset: {}".format(self.asset_memory[0]))

            print("end_total_asset: {}".format(end_total_asset))
            print("total_reward: {}".format(self.state[0] + sum(np.array(self.state[1:(STOCK_DIM + 1)]) * np.array(self.state[(STOCK_DIM + 1):(STOCK_DIM * 2 + 1)]))- self.asset_memory[0]))
            print("total_cost: ", self.cost)
            print("total trades: ", self.trades)

            df_total_value.columns = ["account_value"]
            df_total_value["daily_return"] = df_total_value.pct_change(1)
            sharpe = (4 ** 0.5) * df_total_value["daily_return"].mean() / df_total_value["daily_return"].std()
            print("Sharpe: ", sharpe)

            df_rewards = pd.DataFrame(self.rewards_memory)
            df_rewards.to_csv("results/account_rewards_trade_{}_{}.csv".format(self.model_name, self.iteration))

            # print("total asset: {}".format(self.state[0] + sum(np.array(self.state[1:29]) * np.array(self.state[29:]))))
            # with open("obs.pkl", "wb") as f:
            #     pickle.dump(self.state, f)

            return self.state, self.reward, self.terminal,{}

        else:
            # print(np.array(self.state[1:29]))

            actions = actions * HMAX_NORMALIZE
            # actions = (actions.astype(int))
            if self.turbulence >= self.turbulence_threshold:
                actions = np.array([-HMAX_NORMALIZE] * STOCK_DIM)

            begin_total_asset = self.state[0] + sum(np.array(self.state[1:(STOCK_DIM + 1)]) * np.array(self.state[(STOCK_DIM + 1):(STOCK_DIM * 2 + 1)]))
            # print("begin_total_asset: {}".format(begin_total_asset))

            argsort_actions = np.argsort(actions)

            sell_index = argsort_actions[:np.where(actions < 0)[0].shape[0]]
            buy_index = argsort_actions[::-1][:np.where(actions > 0)[0].shape[0]]

            for index in sell_index:
                # print("take sell action".format(actions[index]))
                self._sell_stock(index, actions[index])

            for index in buy_index:
                # print("take buy action: {}".format(actions[index]))
                self._buy_stock(index, actions[index])

            self.day += 1
            self.data = self.df.loc[self.day,:]
            self.turbulence = self.data["turbulence"].values[0]
            # print(self.turbulence)
            # load next state
            # print("stock_shares: {}".format(self.state[29:]))
            self.state = [self.state[0]] + \
                          self.data.close.values.tolist() + \
                          list(self.state[(STOCK_DIM + 1):(STOCK_DIM * 2 + 1)]) + \
                          self.data.macd.values.tolist() + \
                          self.data.rsi.values.tolist() + \
                          self.data.cci.values.tolist() + \
                          self.data.adx.values.tolist()

            end_total_asset = self.state[0] + sum(np.array(self.state[1:(STOCK_DIM + 1)]) * np.array(self.state[(STOCK_DIM + 1):(STOCK_DIM * 2 + 1)]))
            self.asset_memory.append(end_total_asset)
            #print("end_total_asset: {}".format(end_total_asset))

            self.reward = end_total_asset - begin_total_asset
            # print("step_reward: {}".format(self.reward))
            self.rewards_memory.append(self.reward)

            self.reward = self.reward*REWARD_SCALING

        return self.state, self.reward, self.terminal, {}

    def reset(self):
        self.day = 0
        self.data = self.df.loc[self.day, :]
        self.turbulence = 0
        self.cost = 0
        self.trades = 0
        self.terminal = False
        self.rewards_memory = []

        if self.initial or self.previous_state is None:
            # Trường hợp khởi tạo mới hoặc không có previous_state
            self.asset_memory = [INITIAL_ACCOUNT_BALANCE]
            self.state = [INITIAL_ACCOUNT_BALANCE] + \
                          self.data.close.values.tolist() + \
                          [0] * STOCK_DIM + \
                          self.data.macd.values.tolist() + \
                          self.data.rsi.values.tolist() + \
                          self.data.cci.values.tolist() + \
                          self.data.adx.values.tolist()
        else:
            try:
                previous_total_asset = (
                    self.previous_state[0] +
                    sum(
                        np.array(self.previous_state[1:(STOCK_DIM + 1)]) *
                        np.array(self.previous_state[(STOCK_DIM + 1):(STOCK_DIM * 2 + 1)])
                    )
                )
                self.asset_memory = [previous_total_asset]
                self.state = [self.previous_state[0]] + \
                              self.data.close.values.tolist() + \
                              self.previous_state[(STOCK_DIM + 1):(STOCK_DIM * 2 + 1)] + \
                              self.data.macd.values.tolist() + \
                              self.data.rsi.values.tolist() + \
                              self.data.cci.values.tolist() + \
                              self.data.adx.values.tolist()
            except Exception as e:
                print(f"[WARN] previous_state không hợp lệ, reset lại từ đầu. Chi tiết: {e}")
                self.asset_memory = [INITIAL_ACCOUNT_BALANCE]
                self.state = [INITIAL_ACCOUNT_BALANCE] + \
                              self.data.close.values.tolist() + \
                              [0] * STOCK_DIM + \
                              self.data.macd.values.tolist() + \
                              self.data.rsi.values.tolist() + \
                              self.data.cci.values.tolist() + \
                              self.data.adx.values.tolist()

        return self.state

    def render(self):
        return self.state

    def _seed(self, seed=None):
        self.np_random, seed = seeding.np_random(seed)
        return [seed]
    def save_asset_memory(self):
        return self.asset_memory


In [14]:

def calculate_sharpe_from_csv(csv_path, lookback_days=30):
    """
    Tính Sharpe ratio từ file CSV account_value_validation_<model>.csv
    chỉ lấy 30 ngày gần nhất
    """
    df = pd.read_csv(csv_path, index_col=0)
    df = df.tail(lookback_days)
    df['daily_return'] = df['account_value'].pct_change()
    sharpe = (252 ** 0.5) * df['daily_return'].mean() / df['daily_return'].std()
    return sharpe


In [15]:
A2C_PATH = "/content/results/trained_models/2025-09-02 01:44:37.827718/A2C_30k_dow_378.zip"
PPO_PATH = "/content/results/trained_models/2025-09-02 01:44:37.827718/PPO_100k_dow_378.zip"
TD3_PATH = "/content/results/trained_models/2025-09-02 01:44:37.827718/TD3_10k_dow_378.zip"

In [16]:
import pandas as pd

def preprocess_top30(df, feature_cols, top_n=30):
    """
    Chuẩn hóa data sao cho mỗi ngày có đúng top_n tickers.
    Nếu ngày nào thiếu thì fill từ ngày gần nhất (trước hoặc sau).
    """

    # 🔹 Convert timestamp về dạng int YYYYMMDD
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['timestamp'] = df['timestamp'].dt.strftime("%Y%m%d").astype(int)

    result = []
    all_dates = sorted(df['timestamp'].unique())

    for i, date in enumerate(all_dates):
        day_df = df[df['timestamp'] == date]

        # Lấy top_n tickers (ví dụ dựa vào volume)
        top_df = day_df.nlargest(top_n, 'vol')

        if len(top_df) == top_n:
            result.append(top_df)
        else:
            missing = top_n - len(top_df)
            filled = []

            # Lấy từ ngày trước
            j = i - 1
            while j >= 0 and len(filled) < missing:
                prev_day = result[j]
                candidates = prev_day[~prev_day['ticker'].isin(top_df['ticker'])]
                needed = candidates.head(missing - len(filled))
                filled.append(needed)
                j -= 1

            # Nếu chưa đủ thì lấy từ ngày sau
            if len(filled) < missing:
                k = i + 1
                while k < len(all_dates) and len(filled) < missing:
                    next_day = df[df['timestamp'] == all_dates[k]].nlargest(top_n, 'vol')
                    candidates = next_day[~next_day['ticker'].isin(top_df['ticker'])]
                    needed = candidates.head(missing - len(filled))
                    filled.append(needed)
                    k += 1

            filled_df = pd.concat(filled) if filled else pd.DataFrame(columns=day_df.columns)
            final_day = pd.concat([top_df, filled_df]).head(top_n)
            final_day['timestamp'] = date  # gán timestamp chuẩn
            result.append(final_day)

    df_out = pd.concat(result).sort_values(['timestamp', 'ticker']).reset_index(drop=True)
    df_out.to_csv("visenet/data/output/top_30_stocks_after_train_processed.csv", index=False)
    return df_out


# Example usage
file_path = "/content/top_30_stocks.csv"
df = pd.read_csv(file_path)
feature_cols = ['open','high','low','close','vol','liq','rsi','macd','cci','adx','turbulence']
data_fixed = preprocess_top30(df, feature_cols, top_n=30)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [33]:
!pip install 'shimmy>=2.0'

  Using cached Shimmy-2.0.0-py3-none-any.whl.metadata (3.5 kB)


In [76]:
def data_split(df, start, end):
    """Tách dữ liệu thành tập huấn luyện hoặc kiểm tra dựa trên ngày tháng"""
    data = df[(df.timestamp >= start) & (df.timestamp < end)]
    data=data.sort_values(['timestamp', 'ticker'], ignore_index=True)
    data.index = data.timestamp.factorize()[0]
    return data

In [115]:
import pandas as pd
import numpy as np
from stable_baselines3 import A2C, PPO, TD3
from stable_baselines3.common.vec_env import DummyVecEnv
from datetime import datetime, timedelta
import os

#from env.EnvMultipleStock_trade import StockEnvTrade
from config import config

# ================================================================
# 1. Hàm Sharpe ratio từ account_value
# ================================================================
def calculate_sharpe(account_value_csv):
    df = pd.read_csv(account_value_csv)
    df["daily_return"] = df["account_value"].pct_change(1)
    sharpe = np.sqrt(252) * df["daily_return"].mean() / df["daily_return"].std()
    return sharpe

# ================================================================
# 2. Chọn model tốt nhất trong 3 model
# ================================================================
def pick_best_model():
    results = {}
    for model_name in ["a2c", "ppo", "td3"]:
        csv_file = f"/content/visenet/backtesting/results/{model_name}_account_value_trade.csv"
        if os.path.exists(csv_file):
            sharpe = calculate_sharpe(csv_file)
            results[model_name] = sharpe
            print(f"{model_name} Sharpe = {sharpe:.4f}")
        else:
            print(f"{csv_file} not found, skip {model_name}")
    best_model = max(results, key=results.get)
    print("Best model for today:", best_model)
    return best_model

# ================================================================
# 3. Load model và dự đoán cho hôm nay
# ================================================================
def run_today_strategy(df, today_date):
    """
    df: DataFrame toàn bộ dữ liệu
    today_date: ngày giao dịch hôm nay (int dạng YYYYMMDD)
    """
    best_model_name = pick_best_model()

    # Load model
    if best_model_name == "a2c":
        model = A2C.load(A2C_PATH)
    elif best_model_name == "ppo":
        model = PPO.load(PPO_PATH)
    else:
        model = TD3.load(TD3_PATH)
    
    # Lấy danh sách ngày duy nhất
    unique_days = sorted(df.timestamp.unique())

    # Xác định mốc start và end
    start = unique_days[-31]   # ngày thứ -30
    end = unique_days[-1] + 1  # ngày hôm nay + 1 (để bao gồm hôm nay)

    # Tách dữ liệu trade bằng hàm data_split
    trade = data_split(df, start, end)

    print(len(trade))

    # Tạo env trade
    env_trade = DummyVecEnv([lambda: StockEnvTrade(trade, day=0)])
    obs_trade = env_trade.reset()

    # Dự đoán hành động trong ngày hôm nay
    action, _ = model.predict(obs_trade)
    obs_trade, _, _, _ = env_trade.step(action)

    # Lấy thông tin danh mục sau khi hành động
    try:
        formatted_date = datetime.strptime(str(today_date), "%Y%m%d").strftime("%d-%m-%y")
    except ValueError:
        formatted_date = pd.to_datetime(today_date).strftime("%d-%m-%y")
    strategy = {
        "date": formatted_date,
        "model_used": best_model_name,
        "action": np.array(action).flatten().tolist(),
        "stocks": env_trade.envs[0].df.ticker.unique().tolist(),
        "cash": env_trade.envs[0].state[0],
        "portfolio_value": env_trade.envs[0].state[0] +
                       sum(np.array(env_trade.envs[0].state[1:(STOCK_DIM + 1)]) *
                           np.array(env_trade.envs[0].state[(STOCK_DIM + 1):(STOCK_DIM * 2 + 1)]))
    }

    return strategy

# ================================================================
# 4. Chạy chiến lược hôm nay
# ================================================================
if __name__ == "__main__":
    # Giả sử bạn đã có dataframe df toàn bộ (bao gồm cột timestamp, ticker, open, close...)

    TODAY_DATE = 20250921
    strategy_today = list()
    strategy_today.append(run_today_strategy(data_fixed, TODAY_DATE))
    print("Chiến lược hôm nay:", strategy_today)


a2c Sharpe = 1.2202
ppo Sharpe = 0.2807
td3 Sharpe = 0.6037
Best model for today: a2c
930
Chiến lược hôm nay: [{'date': '21-09-25', 'model_used': 'a2c', 'action': [-0.3561934232711792, -0.21053990721702576, -1.0, 1.0, -0.27515068650245667, 1.0, -0.7952612638473511, -0.1170458197593689, 1.0, -0.37992262840270996, 1.0, 1.0, 0.14170822501182556, 0.7965406179428101, -0.29883211851119995, -0.05433177947998047, 0.5706835985183716, -0.578325629234314, 1.0, 0.24104204773902893, -0.17315205931663513, -0.8194642066955566, -1.0, -1.0, -1.0, 1.0, -0.7298251986503601, 0.9229382872581482, -1.0, 0.19406813383102417], 'stocks': ['ACB', 'BMI', 'CDC', 'CHP', 'DP3', 'DSN', 'EID', 'FOX', 'FPT', 'IDV', 'MCH', 'MPC', 'NCT', 'NTC', 'PGC', 'PHR', 'QNS', 'RAL', 'SBA', 'SHP', 'SNZ', 'TCL', 'TCM', 'TLG', 'VCS', 'VIC', 'VJC', 'VLB', 'VPI', 'VSH'], 'cash': 13149.135000000122, 'portfolio_value': np.float64(1007145.1350000001)}]


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [122]:
def parse_actions(daily_actions):
    alerts = []
    for entry in daily_actions:
        date = entry["date"]
        stocks = entry["stocks"]
        actions = entry["action"]

        signals = []
        for s, a in zip(stocks, actions):
            if a > 0:
                signals.append(f"- ✅ Mua {a} cổ phiếu {s}")
            elif a < 0:
                signals.append(f"- ⚠️ Bán {abs(a)} cổ phiếu {s}")

        # Chỉ soạn mail nếu có tín hiệu
        if signals:
            text = (
                f"📅 Gợi ý chiến lược giao dịch ngày {date}\n\n"
                "Dưới đây là các gợi ý hành động:\n\n"
                + "\n".join(signals)
                + "\n\nChúc bạn giao dịch hiệu quả!\n\n-- Hệ thống VISENET"
            )
            alerts.append(text)

    return alerts


In [124]:
import smtplib
from email.mime.text import MIMEText

def send_email(subject, body, to_email, from_email, app_password):
    # body phải là string
    if isinstance(body, list):
        body = "\n".join(body)

    # Tạo message
    msg = MIMEText(body, "plain", "utf-8")
    msg["Subject"] = subject
    msg["From"] = from_email
    msg["To"] = to_email

    # Gửi mail qua Gmail SMTP
    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
        server.login(from_email, app_password)
        server.send_message(msg)


In [123]:
TO_EMAIL = "trannamhai.5d@gmail.com"
FROM_EMAIL = "visenet2025@gmail.com"
APP_PASSWORD = "wmolzlmjuzkulgpl"
alerts = parse_actions(strategy_today)

send_email(subject="Cảnh báo giao dịch từ VISENET", 
           body=alerts,
           to_email=TO_EMAIL,
           from_email=FROM_EMAIL,
           app_password=APP_PASSWORD)